# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane: content refresh prioritization

I will use the **ranking / scoring** lane: help an editor decide which pages to inspect first for a possible refresh. This is worth pursuing because the starter slice contains 30,000 pages across 32 clients, and a simple queue could focus limited editorial time on pages showing signs of decline. The starter label is an observed snapshot outcome, not a future outcome, so this week is for framing and data checks; later work must define a forward window before claiming prediction.

In [1]:
# This lane is a ranking problem, not a claim that the label is already a future outcome.
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
data_path = next(path for path in candidate_paths if path.exists())
df = pd.read_csv(data_path)

print(f"Loaded {len(df):,} rows and {df['client_id'].nunique()} clients.")
print(f"Content types: {df['content_type'].nunique()}")
print("Starter data loaded successfully.")

Loaded 30,000 rows and 32 clients.
Content types: 3
Starter data loaded successfully.


## 2. The question: decision, action, cost of a wrong call

For a content editor, which pages should be reviewed first for a refresh? The output will be a ranked queue, and the editor will inspect the highest-priority pages and choose an action such as update, monitor, or leave unchanged. A false positive costs an editor's time and may disturb a page that was healthy; a false negative leaves a genuinely declining page unattended. I will therefore judge the eventual queue with **precision@K** at a review capacity chosen with the editor, while comparing it with a transparent rule baseline. The later target must be an observed decline in a future window, not `trend_direction` from this same snapshot.

In [2]:
# The decision metric and leakage exclusions are explicit before modeling.
review_capacity = 100
metric = "precision@K"
leakage_columns = {"trend_direction", "trend_pct", "is_declining_label"}
print(f"Decision metric: {metric} at K={review_capacity:,} pages.")
print(f"Columns reserved from features: {sorted(leakage_columns)}")

Decision metric: precision@K at K=100 pages.
Columns reserved from features: ['is_declining_label', 'trend_direction', 'trend_pct']


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
# Use the raw observed fields for prevalence and scale; do not use the label as a feature.
trend_counts = df["trend_direction"].value_counts(dropna=False)
down_count = int(trend_counts.get("down", 0))
row_count = len(df)

print(f"Observed down-trend pages: {down_count:,} of {row_count:,} ({down_count / row_count:.1%}).")
print(f"Mean impressions over 90 days: {df['impressions_90d'].mean():,.1f}.")
print(f"Mean sessions over 90 days: {df['sessions_90d'].mean():,.1f}.")
print(f"Pages with missing search volume: {df['search_volume'].isna().sum():,}.")
print("These figures motivate prioritization, but they do not measure future predictive performance.")

Observed down-trend pages: 16,262 of 30,000 (54.2%).
Mean impressions over 90 days: 5,200.4.
Mean sessions over 90 days: 37.1.
Pages with missing search volume: 2,468.
These figures motivate prioritization, but they do not measure future predictive performance.


## 3. Quick look at the data (2-3 real numbers)

The starter data has **30,000 pages** across **32 clients**. **16,262 pages (54.2%)** are marked `down` in the observed snapshot, so decline is common enough to justify a prioritization workflow. Average volume is **5,200.4 impressions** and **37.1 sessions** per page over 90 days. These numbers show scale and opportunity, but the decline label is built from the current 30-day comparison and must not be treated as a clean future target.

In [4]:
# Verify the starter target is present, while guarding against the documented label trap.
assert "trend_direction" in df.columns
assert "trend_pct" in df.columns
assert set(["trend_direction", "trend_pct"]).issubset(df.columns)

feature_columns = [column for column in df.columns if column not in leakage_columns]
assert not leakage_columns.intersection(feature_columns)
print(f"Candidate feature columns after exclusions: {len(feature_columns)}")
print("Leakage check passed: trend_direction and trend_pct are excluded from features.")

Candidate feature columns after exclusions: 42
Leakage check passed: trend_direction and trend_pct are excluded from features.


## 4. Careful words: what I can and can't claim

I can claim that the starter slice shows an **observed association** between available page signals and the observed `down` status, and I can build a **directional, decision-support** ranking for editorial review. I can compare that ranking with a transparent baseline using precision@K after defining an honest future label in the warehouse panel. I cannot claim causal impact from a refresh, prove that an edit caused recovery, or claim to predict Google rankings. I also cannot use `trend_direction` or `trend_pct` as features because they define the starter label itself.

The next validation step is to define non-overlapping past feature and future outcome windows, then use client-aware or time-aware evaluation.